# M1 on the bench — pick the bricks out of a real lego pile

A UR3e with a Robotiq 2F-85 and a wrist RealSense, a plywood table, and a pile of real lego parts on
it. This notebook is the orchestrator: it stands the cell up and runs the two submodules in order —
the same two submodules, in the same order, as [`../simulation/main.ipynb`](../simulation/main.ipynb).

| | |
|---|---|
| [`cell.py`](cell.py) | the cell — arm, gripper, camera, table, calibrations. Hardware, no decisions |
| [`submodule_1.py`](submodule_1.py) | look at the pile from two viewpoints, locate every brick view 1 found, stand over the best one |
| [`submodule_2.py`](submodule_2.py) | descend, close, verify, lift, verify again |
| [`submodule_3.py`](submodule_3.py) | the pile perception itself — segmentation, measurement, scoring. Shared with the simulator, unchanged |

**The decision logic is the simulator's, with two deliberate exceptions.** `submodule_1` here is
otherwise line for line its simulation twin — the same survey, the same `PileSession` — because the
simulator is where that logic gets exercised a hundred times an afternoon, and it is only worth
trusting here if it is the same logic. The two places the bench parts company, both because of the
hand-eye calibration and both printed out every run in section 2:

1. **Which brick is grasped is view 1's call alone**, where the simulator requires both views to find
   it. This table is small, so the two viewpoints look across the pile at shallow and very different
   angles, and between that and the calibration's lateral error the two views' centres for one brick
   routinely land further apart than the brick is wide. Requiring agreement there does not filter out
   segmentation accidents, it filters out the whole pile.
2. **Which of the two position estimates is used** — the ray-plane projection here, the triangulation
   in the simulator. Both views are still visited and still triangulated wherever view 2 found the same
   brick; the triangulated point and its ray gap are the rig's own measurement of its calibration.

---

### Before running anything

1. **The table has been touched off.** `python src/tools/calibrate_table.py`. Everything below measures
   heights from that plane; a guess there turns every brick into a clump.
2. **The hand-eye calibration is current** and `config.DEFAULT_CALIBRATION_DIR` points at it. If the
   camera or its mount has been touched since, re-run it.
3. **The robot is in remote control**, the Robotiq URCap is running, and the gripper moves from
   Polyscope.
4. **The e-stop is in reach.** The arm moves for real from section 1 onwards.
5. **`cell.PILE_CENTER` and `submodule_2.DROP_POSITION` match your bench.** They only need to be right
   to a couple of centimetres, but values from somebody else's table aim the camera at bare plywood.

In [ ]:
import contextlib
import os
import sys

import cv2
import matplotlib.pyplot as plt
import numpy as np

SRC_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

from m1.physical import cell as C
from m1.physical import submodule_1, submodule_2
from m1.physical import submodule_3 as perception


def show(image_bgr, title="", width=13):
    """Draw an OpenCV (BGR) image inline."""
    height = width * image_bgr.shape[0] / image_bgr.shape[1]
    plt.figure(figsize=(width, height))
    plt.imshow(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.show()

## 1. Stand up the cell

`build_cell` connects the arm, opens the camera, arms the gripper, loads the hand-eye calibration and
reads the touched-off table plane — everything downstream can then assume it has all of those.

It is a context manager, and a notebook outlives a `with` block, so it is entered into an `ExitStack`
that the last cell closes. **Run that last cell when you are done**, or the camera pipeline and the
RTDE connection stay open and the next run cannot have them.

In [ ]:
stack = contextlib.ExitStack()

cell = stack.enter_context(
    C.build_cell(
        robot_type="ur3e",
        # ip_address=None  -> config.DEFAULT_IP_ADDRESSES["ur3e"]
        camera_resolution="720",
        speed_ratio=10,      # percent of max joint speed; turn it down before turning it up
        linear_speed=0.03,   # m/s for the descent and the lift
        with_gripper=True,
    )
)
print(f"table plane under the pile: z = {cell.table_z_at(*C.PILE_CENTER):+.4f} m")
print(f"jaws {cell.gripper_calibration.min_width * 1000:.0f}-{cell.gripper_calibration.max_width * 1000:.0f} mm, "
      f"currently {cell.finger_width() * 1000:.1f} mm open")

## 2. submodule_1 — find a brick and stand over it

Two viewpoints about 30 cm apart across the pile. Each is perceived on its own; the brick chosen is
the one both views find, in the same place, with the best mean score. **The arm moves.**

Its position then comes from projecting each view's line of sight onto the table plane raised by one
brick height, cross-checked against triangulating the two rays against each other. That is the
opposite way round from the simulation, and section 2's output below says by how much — see
`submodule_1`'s module docstring for why.

In [ ]:
target, views = submodule_1.run(cell)
print()
print("Chosen brick:", target.describe())

### What each view saw

Every brick the perception found is outlined; the five it would send the arm at are numbered in
priority order with the jaw direction drawn across them.

In [ ]:
for view in views:
    show(perception.render_overlay(view.analysis), view.name)

In [ ]:
# The stage-by-stage panel for the first view: height above the table, the foreground the hysteresis
# admitted, the instance labels, and the regions that were dropped with the reason why.
show(perception.render_debug_panel(views[0].analysis), f"{views[0].name} — stages", width=8)

### How the position was arrived at

There is no ground truth here — that is the one thing the bench cannot provide, and the whole reason
for having a simulator. What it *can* provide is the two estimates' disagreement, and that is worth
reading every run: **the numbers below are the hand-eye calibration's error, measured.** They are what
to watch after re-calibrating.

In [ ]:
print(f"  ray-plane projection  {np.round(target.plane_projected, 4)} m   <- used")
print(f"  triangulation         {np.round(target.triangulated, 4)} m")
print(f"  the two views place the brick        {target.view_disagreement * 1000:6.2f} mm apart")
print(f"  the two lines of sight miss by       {target.triangulation_gap * 1000:6.2f} mm")
print(f"  the two methods differ across table  {target.method_disagreement * 1000:6.2f} mm")
print()
print(f"  table under the brick   z={target.table_z:+.4f} m")
print(f"  brick top face          z={target.top_face_z:+.4f} m ({target.height * 1000:.1f} mm tall)")
print(f"  jaws close along        {np.degrees(target.closing_heading):.0f} deg, opened to {target.approach_width * 1000:.1f} mm")

## 3. submodule_2 — grasp it and lift it

Descend the capped distance below the brick's top face, close to a width inside the brick so the pads
stall on it, check, lift, and check again after a pause. Both checks read two signals: the Robotiq's
own object-detection flag, and how far the fingers got compared to how far they were told to go.

**The arm descends into the pile.** Everything is solved and floor-checked first, but this is the cell
to have a hand near the e-stop for.

In [ ]:
result = submodule_2.run(cell, target)
print()
print(result.describe())

In [ ]:
if result.success:
    submodule_2.place(cell, target)
submodule_2.park(cell)

## 4. Emptying the pile — one survey, many picks

Two viewpoints cost the better part of a minute of arm travel and two full pile analyses, and up to
here that price has been paid per *brick*. But the two looks see the whole pile — so
`submodule_1.survey` locates **every** brick they agree on, and `PileSession` serves the picks out of
that map without moving the camera again.

The pile is looked at again only when the map runs low. That is also the moment the bricks that were
buried at the start have become the ones on top, so the occluded ones get located exactly when it is
worth doing — and that repeats until nothing graspable is left.

The price of a cached position is that the pile moves under it. So every pick drops the queued bricks
close enough to the jaws to have been nudged, and a grasp that fails on a position measured *before*
the last pick is remeasured at the next survey rather than written off — while one that fails on a
freshly surveyed position is the brick's own fault, and is left alone for good.

In [ ]:
session = submodule_1.PileSession(cell)
pile_map = session.look()  # the two looks — and a position for every brick in them

print(f"\n{pile_map.remaining} brick(s) located in one pair of looks:\n")
print(f"{'#':>2} {'colour':10s} {'w x l mm':>13s} {'position m':>18s} {'score':>6s} {'views':>8s} {'ray gap':>9s}  source")
for rank, brick in enumerate(pile_map.targets, start=1):
    print(
        f"{rank:2d} {brick.colour:10s} {brick.width * 1000:5.1f} x {brick.length * 1000:5.1f} "
        f"({brick.position[0]:+.4f}, {brick.position[1]:+.4f}) {brick.score:6.3f} "
        f"{brick.view_disagreement * 1000:6.2f}mm {brick.triangulation_gap * 1000:7.2f}mm  {brick.position_source}"
    )

In [ ]:
MAX_ATTEMPTS = 40  # a runaway guard, not a plan: the loop is meant to end when the pile does

picks = []
while (target := session.next_target()) is not None:
    print(
        f"\n{'=' * 78}\npick {len(picks) + 1} — from survey {target.survey_round}, "
        f"{session.map.remaining} brick(s) still queued\n{'=' * 78}"
    )
    try:
        result = submodule_2.run(cell, target)
    except RuntimeError as exception:
        # A pose that stopped being reachable, or a descent the contact guard cut short. Neither is a
        # reason to abandon the pile: record it as a failure and let the session pick the next brick.
        print(f"submodule_2 stopped: {exception}")
        session.record(target, False)
        continue
    if result.success:
        submodule_2.place(cell, target)
    session.record(target, result.success)
    picks.append((target, result))
    if len(picks) >= MAX_ATTEMPTS:
        print("Hit the attempt guard; stopping.")
        break

submodule_2.park(cell)

In [ ]:
print(f"{'colour':12s} {'grasped':>8s} {'survey':>7s} {'views':>9s} {'width':>14s}  reason")
for target, result in picks:
    print(
        f"{target.colour:12s} {'yes' if result.success else 'NO':>8s} {target.survey_round:7d} "
        f"{target.view_disagreement * 1000:6.2f}mm "
        f"{target.width * 1000:5.1f}->{result.width_after_lift * 1000:5.1f}mm  {result.reason}"
    )

stats = session.summary()
successes = sum(1 for _, result in picks if result.success)
saved = 1 - stats["camera_moves"] / max(stats["camera_moves_one_survey_per_pick"], 1)
print(f"\n{successes}/{len(picks)} picked in {cell.elapsed / 60:.1f} min.")
print(
    f"{stats['surveys']} survey(s) for {stats['attempts']} pick(s): {stats['camera_moves']} camera moves "
    f"instead of the {stats['camera_moves_one_survey_per_pick']} a look-every-time loop would take "
    f"— {saved:.0%} of the looking saved."
)
if stats["given_up_on"]:
    print(f"{stats['given_up_on']} brick(s) refused a freshly surveyed grasp and were left where they are.")

## Knobs worth turning

| where | what |
|---|---|
| `C.PILE_CENTER`, `submodule_2.DROP_POSITION` | where the pile is and where picked bricks go. Measure both on your bench |
| `C.VIEWPOINT_JOINT_CONFIGURATIONS` | the two viewpoints, as measured joint configurations. Set an entry to `None` to have `submodule_1.VIEWPOINTS`' Cartesian eye position solved by IK instead, as the simulator does |
| `C.DEFAULT_SPEED_RATIO`, `C.DEFAULT_LINEAR_SPEED` | how fast the arm moves. Down before up |
| `submodule_1.POSITION_SOURCE_PREFERENCE` | which of the two position estimates is used. `"plane_projection"` here, `"triangulation"` in the simulator — flip it after a good hand-eye calibration and watch the disagreement numbers |
| `submodule_1.MATCH_TOLERANCE_M` | how far apart the two views may place the same brick and still be believed. Looser than the simulator's, because both centres carry the calibration's lateral error |
| `submodule_1.RESURVEY_WHEN_REMAINING_BELOW` | how much of each survey to spend before looking again. 1 squeezes every brick out of a map; raising it trades cycle time for fresher positions |
| `submodule_1.FINGER_DISTURBANCE_MARGIN_M` | how far past the open jaws a queued brick is assumed to have been nudged, and so re-located rather than trusted |
| `submodule_1.MAX_CONSECUTIVE_FAILURES` | how many failed grasps in a row before the map is thrown away regardless of what it still has queued |
| `perception.SCORE_WEIGHTS`, `perception.PRIORITY_MIN_CONFIDENCE` | what makes one brick a better grasp than another |
| `submodule_2.GRASP_DEPTH_M`, `GRIPPER_SQUEEZE_M` | how far down the brick's side the pads go, and how hard they pinch |
| `submodule_2.run(..., contact_guard=True)` | arm the UR's contact detection during the descent. A second line of defence behind the touched-off table plane |

## 5. Shut down

Closes the camera pipeline and the RTDE connection. **The gripper is deliberately left as it is** — if
the run ended holding a brick, it is still holding it.

In [ ]:
stack.close()
print("cell closed.")